# A2.1 · "Who is calling?"

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

---

**Risk.** You cannot answer the first question of every investigation.

**Control.** A taxonomy: user-agent vs workload identity, sandboxed vs not, managed vs personal.

**This lab.** Answer 'who is calling?' for every principal in the lab.

| | |
|---|---|
| Open-source tooling | SPIRE |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A2.1"))

"Who is calling?" has a different answer for agents than for people, and most systems can only represent the human one.

In [ ]:
from cybercommons import identity

alice = identity.mint("alice", {"repo:read", "repo:write"})
agent = identity.exchange(alice, "patch-agent", {"repo:write"})

print("token the resource server receives:")
print("  sub   ", agent.sub,   "  ← who the action is *for*")
print("  actor ", agent.actor, "  ← who is actually calling")
print("  act   ", agent.act,   "  ← the chain that got here")
print("  chain ", " → ".join(agent.chain()))
print("  fp    ", agent.fingerprint())

Three distinct identities are in play — the principal, the acting agent, and every intermediary. A system that logs only `sub` cannot answer the question in the title.

In [ ]:
bad = identity.impersonate("alice", "patch-agent", {"repo:write"})
print("with impersonation instead of delegation:")
print("  chain ", " → ".join(bad.chain()))
print("  the agent has vanished. Every log line will say alice did it.")

### Expect

The delegated token shows `alice → patch-agent` with a nested `act` claim. The impersonated token shows only `alice`, with `act = None`.

### Your turn

Check one system you operate: does its audit log have a field for the acting identity that is separate from the principal? If not, every agent action in it is already misattributed.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A2.1.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*